

# Lab 3 · Auditing Your ETL Pipeline
### Does your pipeline see what *you* see?

*Solo lab · about 2 hours · Colab · your own Google Cloud bucket (or Dr. Wanik's backup data)*

Your Craigslist robot has been running on Google Cloud. Tonight you **audit** it: pull your data into this
notebook, **read** the ads, **grade** what the regex extractor got right and wrong, then **fix** the worst field.

| Part | Time | What you do |
|---|---|---|
| **1. Get it running** | 30 min | Finish the Final GCP Guide F26 if you have not yet |
| **2. Search and see** | 25 min | Your cars land in the **Files panel** on the left. Open them, search them, compare to the live ads |
| **3. Grade the robot** | 30 min | A truth table for 5 cars: what the robot extracted vs what the ad really says |
| **4. Fix it** | 30 min | Three ways to pull make and model: plain Python (given), **your regex**, and a typo-fixer (stretch) |
| **5. Wrap up** | 5 min | Save your truth table for A07, answer 3 questions, submit |

**First: File → Save a copy in Drive.** This notebook opens from GitHub, so your edits disappear when you close the tab unless you work in your own copy.

**Pipeline not working tonight?** Set `USE_BACKUP_DATA = True` in Part 2 and use Dr. Wanik's Hartford cars. You can still do the whole lab.

**Stuck?** Ask Claude: paste the cell, the full error, and what you expected.

# Part 1 · Get it running (30 min)

Skip to Part 2 if you already have **5 green checks** and at least one scrape in your bucket.

1. Follow the **[Final GCP Guide F26](https://colab.research.google.com/github/drdave-teaching/OPIM5512-notebooks/blob/main/Module3/FINAL_GCP_Guide_F26.ipynb)** (HuskyCT, Week 3.1) through Step 10.
2. In **Cloud Scheduler**, tick the checkbox, then **Force run**, **one at a time**, waiting for **Success** each time:
   1. `craigslist-scraper-hourly`
   2. `extractor-per-listing-hourly`
   3. `materialize-master-hourly`
3. In **Cloud Storage**, open your bucket: you should see `scrapes/` (raw ads) and `structured/` (the extracted fields).

# Part 2 · Search and see (25 min)

## 2a. Pick your data and load it
- **Your own bucket** (the normal way): set your project ID below. A pop-up asks which Google account to use: pick the
  **same Gmail you used for Google Cloud** (not necessarily your UConn account).
- **Backup** (if your pipeline is not ready): set `USE_BACKUP_DATA = True`. No sign-in needed.

**These three lines are the only ones you edit in Part 2.**

In [ ]:
USE_BACKUP_DATA = False              # True = Dr. Wanik's Hartford sample (no sign-in)
PROJECT_ID = "YOUR-PROJECT-ID"      # e.g. "myscrapers-dww05002-f26-510015"
BUCKET_NAME = "YOUR-PROJECT-ID"     # usually the same as your project ID

In [ ]:
import io
import json
import os
import re
import zipfile
import urllib.request
import pandas as pd

DATA_DIR = "/content/bucket_copy"     # a copy of your bucket, inside this notebook
os.makedirs(DATA_DIR, exist_ok=True)

if USE_BACKUP_DATA:
    SAMPLE_URL = "https://raw.githubusercontent.com/drdave-teaching/OPIM5512-notebooks/main/Module3/data/lab3_hartford_sample.zip"
    zip_bytes = urllib.request.urlopen(SAMPLE_URL).read()
    zipfile.ZipFile(io.BytesIO(zip_bytes)).extractall(DATA_DIR)
    print("Loaded Dr. Wanik's Hartford backup data")
else:
    from google.colab import auth
    from google.cloud import storage
    auth.authenticate_user()
    client = storage.Client(project=PROJECT_ID)
    n_copied = 0
    for blob in client.list_blobs(BUCKET_NAME):
        wanted = blob.name.endswith(".txt") or blob.name.endswith(".csv") or blob.name.endswith(".jsonl")
        if wanted and not blob.name.startswith("state/"):
            local_path = os.path.join(DATA_DIR, blob.name)
            os.makedirs(os.path.dirname(local_path), exist_ok=True)
            blob.download_to_filename(local_path)
            n_copied = n_copied + 1
    print("Copied", n_copied, "files from gs://" + BUCKET_NAME)

cars = pd.read_csv(os.path.join(DATA_DIR, "structured/datasets/listings_master.csv"), dtype={"post_id": str})

link_tables = []
for folder, subfolders, filenames in os.walk(os.path.join(DATA_DIR, "scrapes")):
    for filename in filenames:
        if filename == "index.csv":
            link_tables.append(pd.read_csv(os.path.join(folder, filename), dtype={"post_id": str}))
links = pd.concat(link_tables, ignore_index=True)
links = links[["post_id", "url"]].drop_duplicates("post_id")
cars = cars.merge(links, on="post_id", how="left")
print(len(cars), "cars loaded")

## 2b. Your cars, one file each (look left!)
The next cell makes a folder called **`my_cars`**. Click the **folder icon** in the left sidebar, open `my_cars`, and
**double-click** any file to read it:
- `01_2013_8500_....txt` is the **raw ad** (what the scraper saved)
- `01_2013_8500_....json` is **what the robot extracted** from it (plus the link to the live ad)
- `cars.csv` is the whole table

**Try these two first** (with Dr. Wanik's backup data; with your own cars, hunt for your own examples):
- **File 01** (2013 Sonata): the seller typed `Hyunda Sonata`, and the robot copied it exactly, typo and all. Right by luck, wrong spelling.
- **File 03** (2013 Elantra): the robot says the make is `Contact` and the model is `Information`. Open the `.txt` and find where that came from.

In [ ]:
def read_ad_text(source_txt):
    with open(os.path.join(DATA_DIR, source_txt), encoding="utf-8") as f:
        return f.read()


os.makedirs("/content/my_cars", exist_ok=True)
for i in range(len(cars)):
    row = cars.iloc[i]
    stem = "/content/my_cars/" + str(i + 1).zfill(2) + "_" + str(row["year"]) + "_" + str(row["price"]) + "_" + row["post_id"]
    with open(stem + ".txt", "w", encoding="utf-8") as f:
        f.write(read_ad_text(row["source_txt"]))
    robot = {"post_id": row["post_id"], "url": row["url"], "price": row["price"], "year": row["year"],
             "make": row["make"], "model": row["model"], "mileage": row["mileage"]}
    with open(stem + ".json", "w", encoding="utf-8") as f:
        json.dump(robot, f, indent=2, default=str)
cars.to_csv("/content/my_cars/cars.csv", index=False)
print("Wrote", len(cars), "cars to /content/my_cars (open the folder icon on the left)")
cars[["post_id", "price", "year", "make", "model", "mileage"]].head(10)

**Look at the `make` and `model` columns.** Some look right. Some do not. Keep that thought: it is the mystery for Part 3.

## 2c. Search your cars
The `make` column is not trustworthy yet, so we search the **raw ad text**. Change `SEARCH_WORD` and re-run.
Try a brand (`toyota`, `honda`, `ford`), a feature (`sunroof`, `4wd`), or a phrase like `clean title`.

In [ ]:
SEARCH_WORD = "toyota"

matches = []
for i in range(len(cars)):
    row = cars.iloc[i]
    text = read_ad_text(row["source_txt"])
    if SEARCH_WORD.lower() in text.lower():
        matches.append(row)

print(len(matches), "of", len(cars), "ads mention", repr(SEARCH_WORD))
for row in matches:
    print(row["price"], row["year"], row["mileage"], row["url"])

## 2d. See it with your own eyes
Pick **3 cars**. Open each one's `.txt` file on the left **and** its link in a new tab. Can you find the price, the year,
the make and model, and the odometer in both?

**Write 2 or 3 sentences:** what does the raw text keep from the web page, and what does it lose?

*Your answer:*

# Part 3 · Grade the robot (30 min)

The extractor (Module 3.2) uses regular expressions to turn each raw ad into `price, year, make, model, mileage`.
How good is it? You find out with a **truth table**: what each ad **really** says, typed by you.

## 3a. Write down the truth for 5 cars
Run the cell, then open each car's file (or link) and read the real values. Near the top of each ad there is a line
that is just the **year**, and the next line is the **make and model** (for example `2011` then `Honda Accord LX`),
and further down `odometer:` then the miles.

**Rules for the truth table:**
- numbers without `$` or commas (`4450`, not `"$4,450"`)
- make and model in **lowercase**, model = **first word only** (`honda`, `accord`)
- write the **real** make: if the seller typed `Hyunda`, the truth is `hyundai`

In [ ]:
graded = cars.head(5)

for i in range(len(graded)):
    row = graded.iloc[i]
    print("file", str(i + 1).zfill(2), "|", row["post_id"], "| robot says:", row["price"], row["year"], row["make"], row["model"], row["mileage"])

print()
print("Copy this into the next cell and fill in the TRUE values:")
print("truth = {")
for i in range(len(graded)):
    pid = graded.iloc[i]["post_id"]
    print('    "' + pid + '": {"price": None, "year": None, "make": "", "model": "", "mileage": None},')
print("}")

In [ ]:
# Paste the dictionary from above and type the TRUE values from each ad.
truth = {
}

## 3b. Score the robot

In [ ]:
FIELDS = ["price", "year", "make", "model", "mileage"]


def same(robot_value, true_value, field):
    if field in ["make", "model"]:
        return str(robot_value).strip().lower() == str(true_value).strip().lower()
    try:
        return float(robot_value) == float(true_value)
    except (TypeError, ValueError):
        return False


def score(table, truth):
    rows = []
    for field in FIELDS:
        right = 0
        for pid in truth:
            table_row = table[table["post_id"] == pid].iloc[0]
            if same(table_row[field], truth[pid][field], field):
                right = right + 1
        rows.append({"field": field, "correct": right, "out_of": len(truth), "accuracy": right / max(len(truth), 1)})
    return pd.DataFrame(rows)


before = score(cars, truth)
print(before.to_string())

**Which fields does the robot nail? Which does it miss?**

## 3c. The mystery: what is the make/model regex actually grabbing?
Here is the rule, straight from `cloud_function/extractor-per-listing/main.py`:

```python
MAKE_MODEL_RE = re.compile(r"\b([A-Z][a-z]+)\s+([A-Z][A-Za-z0-9]+)")
```

In words: *the **first** place on the page where one Capitalized word is followed by another Capitalized word.*
Run the cell to see what it grabs from each of your 5 cars.

In [ ]:
MAKE_MODEL_RE = re.compile(r"\b([A-Z][a-z]+)\s+([A-Z][A-Za-z0-9]+)")

for i in range(len(graded)):
    text = read_ad_text(graded.iloc[i]["source_txt"])
    found = MAKE_MODEL_RE.search(text)
    first_line = text.split("\n")[0]
    print("file", str(i + 1).zfill(2), "| grabs:", found.groups() if found else None, "| page title:", first_line[:70])

Each ad's text starts with the page title the seller typed. If they Capitalized it (`2011 Honda Accord`), the rule
works. If they typed it in lowercase or ALL CAPS, the first two Capitalized words are somewhere else on the page:
a button (`Contact Information`), or even the **town** (`Vernon Rockville`).

The reliable spot is the details block near the photos. Print it for one car and look closely (`repr` shows each line exactly):

In [ ]:
text = read_ad_text(graded.iloc[0]["source_txt"])
ad_lines = text.split("\n")
for i in range(len(ad_lines)):
    if ad_lines[i].lower().startswith("odometer"):
        start = max(0, i - 14)
        for j in range(start, i + 2):
            print(repr(ad_lines[j]))
        break

# Part 4 · Fix it: three ways to get make and model (30 min)

The clue: a line that is **only a 4-digit year**, and the **next line** starts with the **make** then the **model**.
Three ways to use that clue, from easiest to fanciest.

## Way 1 · Plain Python, no regex (given: read it, run it)
Walk the lines. When a line is exactly 4 digits and starts with 19 or 20, split the next line into words.

In [ ]:
def way1_make_model(text):
    ad_lines = text.split("\n")
    for i in range(len(ad_lines) - 1):
        line = ad_lines[i].strip()
        if len(line) == 4 and line.isdigit() and (line.startswith("19") or line.startswith("20")):
            words = ad_lines[i + 1].strip().lower().split()
            if len(words) >= 2:
                return words[0], words[1]
    return "", ""


print(way1_make_model(read_ad_text(graded.iloc[0]["source_txt"])))

## Way 2 · Your regex: the same idea in ONE line
Write one regular expression that matches the year line, the line break, then captures the **make** (group 1) and the
**model** (group 2). Build it a piece at a time and test at **regex101.com** (pick the Python flavor).

| Piece | Means |
|---|---|
| `^` and `$` with `re.MULTILINE` | start / end of a **line** |
| `(?:19\|20)\d{2}` | a year like 1998 or 2017 (`(?: )` groups without capturing) |
| `\n` | the line break |
| `([A-Za-z-]+)` | capture one word of letters (and hyphens, for `mercedes-benz`) |
| `\s+` | one or more spaces |

<details><summary><b>Hint 1</b> (click)</summary>

Start with just the year line: `^(?:19|20)\d{2}$` matches a line like `2013`. Try it on regex101 with one of your ads.
</details>

<details><summary><b>Hint 2</b></summary>

Replace the `$` with `\n` so the match continues onto the next line: `^(?:19|20)\d{2}\n`
</details>

<details><summary><b>Hint 3</b></summary>

Now capture the first word of the next line: `^(?:19|20)\d{2}\n([A-Za-z-]+)`. You add the space and the second group.
</details>

In [ ]:
BETTER_MAKE_MODEL_RE = re.compile(r"YOUR PATTERN HERE", re.MULTILINE)   # <-- your one line


def way2_make_model(text):
    found = BETTER_MAKE_MODEL_RE.search(text)
    if found is None:
        return "", ""
    return found.group(1).lower(), found.group(2).lower()


print(way2_make_model(read_ad_text(graded.iloc[0]["source_txt"])))

## Way 3 · Stretch: fix the typos too
Neither Way 1 nor Way 2 can know that `hyunda` means `hyundai`. A short list of real makes plus Python's `difflib`
(which finds the closest spelling) can. Run it, then read the table.

In [ ]:
import difflib

KNOWN_MAKES = ["acura", "audi", "bmw", "buick", "cadillac", "chevrolet", "chrysler", "dodge", "fiat", "ford", "gmc",
               "honda", "hyundai", "infiniti", "jaguar", "jeep", "kia", "land", "lexus", "lincoln", "mazda",
               "mercedes-benz", "mercury", "mini", "mitsubishi", "nissan", "pontiac", "porsche", "ram", "saab",
               "saturn", "scion", "subaru", "tesla", "toyota", "volkswagen", "volvo"]
NICKNAMES = {"chevy": "chevrolet", "vw": "volkswagen", "mercedes": "mercedes-benz"}


def way3_make_model(text):
    make, model = way1_make_model(text)
    if make in NICKNAMES:
        make = NICKNAMES[make]
    closest = difflib.get_close_matches(make, KNOWN_MAKES, n=1, cutoff=0.75)
    if len(closest) > 0:
        make = closest[0]
    return make, model


print(way3_make_model(read_ad_text(graded.iloc[0]["source_txt"])))

## 4b. Re-grade: the robot vs your three ways

In [ ]:
def graded_with(make_model_function):
    fixed = graded.copy()
    for i in range(len(fixed)):
        text = read_ad_text(fixed.iloc[i]["source_txt"])
        make, model = make_model_function(text)
        fixed.iloc[i, fixed.columns.get_loc("make")] = make
        fixed.iloc[i, fixed.columns.get_loc("model")] = model
    return fixed


comparison = before[["field", "accuracy"]].rename(columns={"accuracy": "robot"})
comparison["way1_python"] = score(graded_with(way1_make_model), truth)["accuracy"]
comparison["way2_your_regex"] = score(graded_with(way2_make_model), truth)["accuracy"]
comparison["way3_typo_fix"] = score(graded_with(way3_make_model), truth)["accuracy"]
print(comparison.to_string())

**Stretch:** run your regex on **all** your cars. How many ads does it miss completely? Open one: why?

In [ ]:
missed = []
for i in range(len(cars)):
    make, model = way2_make_model(read_ad_text(cars.iloc[i]["source_txt"]))
    if make == "":
        missed.append(cars.iloc[i]["url"])
print(len(missed), "of", len(cars), "ads had no match")
for url in missed:
    print(url)

# Part 5 · Wrap up (5 min)

## Save your truth table (you will use it again in A07)
In A07 you let Gemini read the ads instead of a regex. You will score it on **these same 5 cars**, so save your truth
table now: to your bucket (if you used your own data) and to a file you can download.

In [ ]:
with open("/content/my_cars/truth_table.json", "w", encoding="utf-8") as f:
    json.dump(truth, f, indent=2)
print("Saved /content/my_cars/truth_table.json (right-click it on the left and Download)")

if not USE_BACKUP_DATA:
    client.bucket(BUCKET_NAME).blob("lab3/truth_table.json").upload_from_string(json.dumps(truth, indent=2), content_type="application/json")
    print("Also saved to gs://" + BUCKET_NAME + "/lab3/truth_table.json")

## Three short answers (1 or 2 sentences each)
1. Which field did the robot get **most** right, and which **least**? Why?
2. Where could **your** regex still break? (Think about how different people post their ads.)
3. In A07, Gemini reads the ad instead of a regex. What will it do **better** than Ways 1 to 3, and what could go **wrong**?

*Your answers:*

---

**Submit:** File → Download → `.ipynb`, then upload it to **Submit Lab 3** on HuskyCT (due Thu Oct 8, 11:59 PM).
Keep your pipeline running: the more hours it scrapes, the better your data for the midterm.